In [ ]:
import lab as B
import torch
import matplotlib.pyplot as plt
import os
import numpy as np
import neuralprocesses.torch as nps
from neutdes.datasets import generate_mock_data

In [ ]:
device = torch.device('cpu')

if torch.cuda.is_available():
    print("Setting device to the gpu to `cuda`")
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    print("Setting device to the gpu to `mps`")
    device = torch.device("mps")

    # Required or else the we may run out of memory
    os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

B.set_global_device(device)
torch.set_default_dtype(torch.float32)
B.set_random_seed(0)

In [ ]:
# Construct a ConvCNP.
convcnp = nps.construct_agnp(
    dim_x=1,
    dim_y=1,
    likelihood="het",
    # dim_lv=16,
    # points_per_unit=16,
    dtype=torch.float32
)
convcnp = convcnp.to(device)

# Construct optimiser.
opt = torch.optim.Adam(convcnp.parameters(), 3e-4)
losses = []

for i in range(10000):
    # Sample a batch of new context and target sets. Replace this with your data. The
    # shapes are `(batch_size, dimensionality, num_data)`.
    x, y = generate_mock_data(amplitude_range=(0.9, 1.1),
                              x_break_range=(95, 105),
                              alpha_1_range=(-2.5, -1.5),
                              alpha_2_range=(1.5, 2.25),
                              delta_range=(0.09, 1.1),
                              batch_size=32, 
                              num_points=200, 
                              device=device)

    inds = np.random.permutation(x.shape[2])
    xc, yc = x[:, :, inds[:80]], y[:, :, inds[:80]]
    xt, yt = x[:, :, inds[20:]], y[:, :, inds[20:]]

    # Compute the loss and update the model parameters.
    loss = -torch.mean(nps.loglik(convcnp, xc, yc, xt, yt, normalise=True, dtype_lik=torch.float32))
    losses.append(loss.item())
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

In [ ]:

# Make predictions on some new data
pxt, pyt = generate_mock_data(amplitude_range=(0.9, 1.1),
                              x_break_range=(95, 105),
                              alpha_1_range=(-2.5, -1.5),
                              alpha_2_range=(1.5, 2.25),
                              delta_range=(0.09, 1.1),
                              batch_size=1, num_points=100, device=device)

# Use batch to create random set of context points
inds = np.random.permutation(pxt.shape[2])
pxc, pyc = pxt[:1, :, inds[:15]], pyt[:1, :, inds[:15]]
pxc, pyc = pxt[:1, :, :15], pyt[:1, :, :15]

# Running without moving back to the cpu, or simply using `ar_predict` either crashes
# the system or runs out of memory...
with B.on_device('cpu'):
    mean, var, noiseless_samples, noisy_samples = nps.predict(convcnp.cpu(), pxc.cpu(), pyc.cpu(), pxt.cpu())
    # mean, var, noiseless_samples, noisy_samples = nps.ar_predict(convcnp.cpu(), pxc.cpu(), pyc.cpu(), pxt.cpu())

# mean, var, noiseless_samples, noisy_samples = nps.ar_predict(convcnp, pxc, pyc, pxt)

In [ ]:
import seaborn as sns

sns.set_style("ticks")
sns.set_context("talk", font_scale=1.5)

f, ax = plt.subplots(figsize=(12, 8), layout='constrained')

for i in range(1):
    ctx_x, ctx_y = pxc.cpu().numpy()[i, 0, :], pyc.cpu().numpy()[i, 0, :]
    tar_x, tar_y = pxt.cpu().numpy()[i, 0, :], pyt.cpu().numpy()[i, 0, :]

    ax.scatter(ctx_x, ctx_y)
    ax.scatter(tar_x, tar_y, color="none", edgecolor="k")

for i in range(1):
    pre_x, pre_y = pxt.cpu().numpy()[i, 0, :], mean.detach().cpu().numpy()[i, 0, :]
    pre_yerr = np.sqrt(var.detach().cpu().numpy()[i, 0, :])
    
    ax.plot(pre_x, pre_y)
    ax.fill_between(pre_x, pre_y - pre_yerr, pre_y + pre_yerr, alpha=0.25, color="C0")

ax.set_ylabel("Normalized Flux")
ax.set_xlabel("Time Since Disruption [MJD]")